# 03 — Job Recommender (Unsupervised)

Build a content-based recommender using **TF-IDF + Cosine Similarity**.

In [ ]:
import sys
sys.path.insert(0,"..")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")
%matplotlib inline

## 1. Load Job Corpus

In [ ]:
from src.config import JOB_CORPUS_CSV
from src.data.load_data import load_jobs
from src.data.preprocess import build_job_corpus

try:
    job_corpus = pd.read_csv(JOB_CORPUS_CSV)
    print("Loaded from cache:", job_corpus.shape)
except FileNotFoundError:
    jobs       = load_jobs()
    job_corpus = build_job_corpus(jobs)
    print("Built fresh:", job_corpus.shape)

## 2. Build TF-IDF Job Index

Vectorize all job descriptions and save the matrix to disk.

In [ ]:
from src.models.recommender import build_job_index
vectorizer, job_matrix = build_job_index(job_corpus)
print(f"Job matrix: {job_matrix.shape}")

## 3. Recommend Jobs for a Sample Resume

In [ ]:
from src.data.preprocess import clean_text
from src.models.recommender import recommend_jobs

sample_resume = """
Data Scientist with expertise in Python, pandas, scikit-learn, machine learning,
statistical modelling, SQL, and data visualization. Experience with NLP and
predictive analytics in a financial services environment.
"""

clean = clean_text(sample_resume)
recommendations = recommend_jobs(
    clean, job_corpus, vectorizer=vectorizer, job_matrix=job_matrix, top_n=10
)
print(recommendations[["rank","title","company","match_score"]].to_string())

## 4. Match Score Distribution

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
resume_vec = vectorizer.transform([clean])
all_scores = cosine_similarity(resume_vec, job_matrix).flatten() * 100

plt.figure(figsize=(10,4))
plt.hist(all_scores, bins=50, color="#6366f1", alpha=0.8, edgecolor="white")
plt.axvline(all_scores.mean(), color="#f472b6", linestyle="--", label=f"Mean: {all_scores.mean():.1f}%")
plt.xlabel("Match Score (%)")
plt.ylabel("Number of Jobs")
plt.title("Distribution of Cosine Similarity Scores")
plt.legend()
plt.tight_layout()
plt.savefig("../reports/figures/match_score_dist.png", dpi=150)
plt.show()

## 5. Top-10 Matches Visualised

In [ ]:
fig, ax = plt.subplots(figsize=(10,5))
titles = recommendations["title"].tolist()[::-1]
scores = recommendations["match_score"].tolist()[::-1]
colors = ["#34d399" if s>=60 else "#fbbf24" if s>=35 else "#ef4444" for s in scores]
ax.barh(titles, scores, color=colors)
ax.set_xlabel("Match Score (%)")
ax.set_title("Top-10 Job Recommendations")
for i, (s, t) in enumerate(zip(scores, titles)):
    ax.text(s+0.5, i, f"{s:.1f}%", va="center", fontsize=9)
plt.tight_layout()
plt.savefig("../reports/figures/top10_recommendations.png", dpi=150)
plt.show()

## 6. Precision@K Evaluation (qualitative check)

In [ ]:
# Manually label top-10 as relevant (1) or not (0) based on your domain knowledge
# Then compute Precision@K
from src.evaluate import precision_at_k

# Example: assuming first 7 of top-10 are relevant for a Data Scientist
recommended_titles = recommendations["title"].tolist()
relevant_titles    = ["Data Scientist", "ML Engineer", "Data Analyst",
                      "AI Researcher", "Data Science Lead",
                      "Senior Data Scientist", "Research Scientist"]

p_at_5  = precision_at_k(recommended_titles, relevant_titles, k=5)
p_at_10 = precision_at_k(recommended_titles, relevant_titles, k=10)
print(f"Precision@5:  {p_at_5:.3f}")
print(f"Precision@10: {p_at_10:.3f}")

## ✅ Recommender complete. Next → 04_clustering_topics.ipynb